# Segment 4 · Robot learning with a surgical VLA and a world model
**CoRL 2026 Tutorial — Robot Learning on the Path to Surgical Autonomy** · minutes 45–60 · *v4*

In Segment 3 a small policy (ACT) learned SutureBot from scratch. Here we ask what large pretrained models add:

| | ACT (Segment 3) | **GR00T-H-N1.7** (VLA) | **Cosmos-H-Dreams** (world model) |
|---|---|---|---|
| Input | images + robot state | images + **language instruction** | one frame + a sequence of actions |
| Output | action chunk | action chunk (50 steps @ 30 Hz ≈ 1.7 s) | **video** of what those actions would do |
| Pretraining | none | Open-H: 20 robot platforms, 770 h | Open-H, 9 embodiments |
| Role today | baseline, compared in sections 3–4 | the policy | a "simulator" to look at actions, then to run the policy in |

**How this notebook works**
- ▶ **Main thread** — the cells we run together. They call a shared GPU server, so this notebook runs on a free CPU Colab.
- 🧪 **Explore on your own** — the section at the bottom. Every cell there is independent; run any of them whenever you like.
- If the server is unreachable, set `BACKEND = "cache"` and every main-thread cell replays pre-computed results.

Models: [GR00T-H-N1.7](https://huggingface.co/nvidia/GR00T-H-N1.7) · [Cosmos-H-Dreams](https://huggingface.co/nvidia/Cosmos-H-Dreams) · Data: [Open-H-Embodiment / SutureBot](https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-Open-H-Embodiment)

## 📖 Background · Meet the two models
*Reading material only, nothing to run. Skim it now, or come back to it after the session.*

### GR00T-H-N1.7: a vision-language-action (VLA) policy
**What it is.** NVIDIA's surgical robot foundation model (the "H" stands for healthcare). It looks at camera images, reads a short text instruction, and outputs the next robot motions. It has about 3 B parameters, was pretrained on the Open-H-Embodiment dataset, and the checkpoint we use has been fine-tuned on data that includes SutureBot.

**How it works.** Two parts, trained together:
1. **Vision-language backbone** (Cosmos-Reason2-2B). It turns the camera images and the instruction into a set of features that describe the scene and the goal.
2. **Action head**, a diffusion transformer trained with *flow matching*. It starts from random noise and refines it over a few steps into a full **action chunk**, using the backbone's features as a guide.

Predicting a whole chunk (50 steps ≈ 1.7 s) at once, instead of one step at a time, gives smoother motion and fewer model calls. ACT from Segment 3 uses the same idea.

**One model, many robots.** An **embodiment tag** tells the model which robot it is driving, so it knows which cameras to expect and how to read and write that robot's states and actions. We use `jhu_imerse_dvrk` (the dVRK-Si setup used for SutureBot):

| | |
|---|---|
| Cameras | `endoscope_left`, `wrist_left`, `wrist_right` |
| Instruction | the subtask: *needle pickup*, *needle throw* or *knot tying* |
| Robot state | 16 values, but trained with 100 % dropout for this robot, so it is ignored |
| Output | 50 steps × 16 values at 30 Hz. Per arm (PSM1, PSM2): position xyz (m), orientation quaternion (xyzw), jaw angle (rad) |

Inside, the model predicts motion *relative* to the current pose (xyz plus a 6-D rotation). It converts back to absolute values in the robot's own units before returning them.

**Why it matters.** In the Open-H report, GR00T-H matched ACT's SutureBot success rate with only a third of the training data. With all the data it reached ≈73 % against ACT's ≈50 %, and it was the only policy to complete end-to-end suturing (25 %).

---

### Cosmos-H-Dreams: an action-conditioned world model
**What it is.** A video model that answers one question: *"If the robot made these motions from this exact scene, what would the endoscope see?"* Give it one real frame and a sequence of robot actions, and it generates the video that should follow.

**How it works.**
- It is a fast, **distilled** "student" of Cosmos-H-Surgical-Simulator, which is built on Cosmos-Predict2.5-2B. Distillation trains a small, quick model to copy a large, slow one. On our server one 72-frame rollout takes a few seconds.
- It generates **autoregressively in blocks**: context frame (288 × 512) + the next actions → 12 new frames (~1.2 s at 10 fps). The newest frames then become the context for the next block. Small errors can build up from block to block, so we stop after 6 blocks (72 frames, ~7 s), the same length used in the Open-H report.
- It only renders the **left endoscope view**. It does not render the wrist cameras.

**Action format.** Cosmos-H-Dreams uses one shared 44-D action space for all the robots it supports. The dVRK uses 20 of those dimensions: for each arm, xyz (3) + 6-D rotation (6) + jaw (1). These are measured *relative to the robot's pose at the start frame* and normalized. Our server converts any absolute 16-D, 30 Hz action array (from GR00T-H, the recording, ACT or your own policy) into this format and keeps every 3rd step to match 10 fps. That is why any policy's output can be "dreamed".

**What it's good for.**
- **Looking at a policy's actions without a robot.** Action curves are hard to read. A video shows whether the needle actually moves where it should.
- **Testing a policy safely.** Feed the generated frames back to the policy and repeat (closed loop). NVIDIA used this approach (Cosmos-Surg-dVRK) to screen GR00T-H checkpoints; see Explore F.

**What to watch out for.** It is a learned model, not a physics engine. It can blur tissue, lose track of the thread, or show physically impossible results. That is why every comparison below also rolls out the **recorded actions**: whatever gap remains between that rollout and the real video is the world model's own error. On our held-out clips, recorded-action rollouts score about L1 0.02 / SSIM 0.85 against the real video, compared with about 0.04 / 0.77 for a frozen first frame.

---

### How they fit together today
```
 real frame + instruction ──► GR00T-H-N1.7 ──► 50-step action chunk ──┐
                                                                      ├──► Cosmos-H-Dreams ──► predicted video ──► compare with real video
 recorded actions (reference) ────────────────────────────────────────┘                                            (L1, SSIM)
```
- **GR00T-H** is scored *open loop* against the recorded actions (section 3): how close are its motions to the surgeon's?
- **Cosmos-H-Dreams** turns those motions into video (section 4): what would they *do* to the scene?

**Quick glossary**
- **Open loop:** the policy always sees the real recorded scene, so its mistakes never affect what it sees next. **Closed loop:** it sees the results of its own actions, so mistakes can build up.
- **Action chunk:** a block of future actions predicted in one call.
- **PSM:** Patient Side Manipulator, one of the dVRK's instrument arms (PSM1 and PSM2).
- **L1 / SSIM:** per-frame pixel error (lower is better) and structural similarity (higher is better) between generated and real video.

## ▶ 0 · Setup (≈1 min)

In [ ]:
#@title Connect to the tutorial server { display-mode: "form" }
API_URL = "https://corl-seg4-5xptyrp9u.gobrev.dev"   #@param {type:"string"}
API_KEY = ""                          #@param {type:"string"}
BACKEND = "api"                       #@param ["api", "cache"]
CACHE_URL = "https://huggingface.co/datasets/kdb6df/corl2026-seg4-cache/resolve/main/seg4_bundle_{episode}.zip"  #@param {type:"string"}
ACT_URL = "https://huggingface.co/kdb6df/corl2026-suturebot-act/resolve/main/predictions/act_{episode}.npz"  #@param {type:"string"}
EPISODE = 0                           #@param [0, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451] {type:"raw"}
FRAME = -1                            #@param {type:"integer"}
EPISODE, FRAME = int(EPISODE), int(FRAME)   # Colab can pass form values as text
# EPISODE 0 = the demo clip (1441). 1441-1444 are needle throws, 1445-1451 knot tying; all are held out
# from both models and pre-computed. FRAME -1 = that clip's pre-computed start frame (returns instantly).

!pip -q install scikit-image imageio[ffmpeg] av huggingface_hub pyarrow jinja2 > /dev/null

import json, time, io, base64, os, numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from IPython.display import HTML, display
H = {"X-API-Key": API_KEY}

def _get(path):  return requests.get(API_URL + path, headers=H, timeout=60)
def _post(path, j, timeout=120):
    r = requests.post(API_URL + path, json=j, headers=H, timeout=timeout); r.raise_for_status(); return r.json()

if BACKEND == "api":
    try:
        h = requests.get(API_URL + "/health", timeout=10).json()
        print("✅ server up:", h["model"], "| episodes:", h["episodes"], "| cosmos queue:", h["cosmos_queue"])
        if EPISODE == 0: EPISODE = h["episodes"][0]
        if FRAME < 0: FRAME = int(h.get("start_frames", {}).get(str(EPISODE), 0))
    except Exception as e:
        print("⚠️ server unreachable -> switching to cached results.", e); BACKEND = "cache"

BUNDLE = None
if BACKEND == "cache":
    import urllib.request, zipfile
    if EPISODE == 0: EPISODE = 1441
    urllib.request.urlretrieve(CACHE_URL.format(episode=EPISODE), "bundle.zip")
    import shutil; shutil.rmtree("bundle", ignore_errors=True); zipfile.ZipFile("bundle.zip").extractall("bundle")
    BUNDLE = json.load(open("bundle/bundle.json")); EPISODE, FRAME = BUNDLE["episode"], BUNDLE["frame"]
print("backend:", BACKEND, "| episode:", EPISODE, "| start frame:", FRAME)

## ▶ 1 · The clip (≈1.5 min)
One **held-out test episode** from SutureBot (dVRK-Si, JHU). Neither model saw it in training. The default, 1441, is the same episode as Segment 3, so the ACT numbers are directly comparable. Want another? Pick any of 1441–1451 in the Setup cell and re-run from there: every one is pre-computed.

In [ ]:
#@title Download just this episode (parquet + endoscope video) { display-mode: "form" }
from huggingface_hub import hf_hub_download
REPO, ROOT = "nvidia/PhysicalAI-Robotics-Open-H-Embodiment", "Surgical/jhu/imerse/suturebot"
c = EPISODE // 1000
pq  = hf_hub_download(REPO, f"{ROOT}/data/chunk-{c:03d}/episode_{EPISODE:06d}.parquet", repo_type="dataset")
vid = hf_hub_download(REPO, f"{ROOT}/videos/chunk-{c:03d}/observation.images.endoscope.left/episode_{EPISODE:06d}.mp4", repo_type="dataset")
df = pd.read_parquet(pq)
GT = np.stack(df["action"].values).astype(np.float32)          # (T, 16) absolute @ 30 Hz
NAMES = [f"psm{a}_{k}" for a in (1, 2) for k in ("x","y","z","qx","qy","qz","qw","jaw")]
POS, JAWS = [0, 1, 2, 8, 9, 10], [7, 15]                    # x/y/z (m) and jaw (rad) columns
def unit(d):  return {"x": "mm", "y": "mm", "z": "mm", "jaw": "rad"}.get(NAMES[d].split("_")[1], "")
def scale(d): return 1000 if unit(d) == "mm" else 1            # plot positions in mm
TASKS = {0: "knot tying", 1: "needle pickup", 2: "needle throw"}
print(f"{len(GT)} frames ({len(GT)/30:.1f} s) | task: {TASKS[int(df.task_index.iloc[0])]}")
if "instruction.text" in df: print("instruction:", df["instruction.text"].iloc[0])

import av
def read_frames(path, idx=None, size=(512, 288)):
    out = []
    with av.open(path) as f:
        for i, fr in enumerate(f.decode(video=0)):
            if idx is None or i in idx: out.append(fr.to_ndarray(format="rgb24", width=size[0], height=size[1]))
            if idx is not None and i > max(idx): break
    return np.stack(out)

def show_video(path, width=640):
    if path is None: return
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width={width} controls loop autoplay muted src="data:video/mp4;base64,{b64}"></video>'))

def to_mp4(frames, path, fps=10):
    import imageio.v3 as iio; iio.imwrite(path, frames, fps=fps, codec="libx264"); return path

show_video(to_mp4(read_frames(vid, size=(480, 270))[::3], "clip.mp4"))

# Segment 3's ACT, precomputed on the server's GPU (server/act_precompute.py): see section 3
try:
    _z = np.load(io.BytesIO(requests.get(ACT_URL.format(episode=EPISODE), timeout=60).content))
    ACT = {k: _z[k] for k in _z.files}
    print(f"ACT predictions loaded (start frame {int(ACT['start_frame'])})")
except Exception as e:
    ACT = None; print("⚠️ ACT predictions unavailable, sections 3-4 show GR00T-H only:", e)

## ▶ 2 · Ask GR00T-H for an action chunk (≈3 min)
**In:** three camera views (endoscope + two wrist cameras) and a text instruction. (For this dVRK embodiment the model was trained with 100 % state dropout, so it does *not* use joint state.)
**Out:** 50 future steps × 16 values (xyz + quaternion + jaw for each arm), in the robot's own units.

In [ ]:
#@title GR00T-H inference { display-mode: "form" }
if BACKEND == "api":
    r = _post("/v1/groot/act", {"episode": EPISODE, "frame": FRAME})
else:
    r = BUNDLE["act"]
CHUNK = np.array(r["actions"])
print(f"instruction: {r['instruction']!r} | chunk {CHUNK.shape} | latency {r.get('latency_ms','cached')} ms | cached={r.get('cached')}")

t = np.arange(FRAME, FRAME + len(CHUNK)) / 30
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
for j, (lab, dims) in enumerate([("PSM1 position", [0,1,2]), ("PSM2 position", [8,9,10]), ("jaws", [7,15])]):
    for d in dims:
        ax[j].plot(t, scale(d) * GT[FRAME:FRAME+len(CHUNK), d], "k-", lw=1, alpha=.6)
        ax[j].plot(t, scale(d) * CHUNK[:, d], "--", label=NAMES[d])
    ax[j].set_title(lab); ax[j].set_xlabel("time (s)"); ax[j].set_ylabel(unit(dims[0])); ax[j].legend(fontsize=7)
fig.suptitle("GR00T-H prediction (dashed) vs what the surgeon-operator actually did (black)"); plt.tight_layout()

## ▶ 3 · Open-loop evaluation, the same way NVIDIA does it, GR00T-H vs ACT (≈3 min)
We slide over the whole episode: every 50 steps, give GR00T-H the **real** observation, keep its 50-step chunk, and compare the stitched prediction with the recorded actions.
Metric = mean squared / absolute error over all 16 dimensions and all steps, exactly as in GR00T-H's official [`open_loop_eval.py`](https://github.com/NVIDIA-Medtech/GR00T-H/blob/main/gr00t/eval/open_loop_eval.py).
This is **open-loop**: errors never compound, because the model always sees the real scene. It says little about closed-loop success (see the Open-H report's discussion).
The **hold pose** row is a no-model reference (repeat the current pose for 50 steps). Surgical motion is slow, so it is a strong baseline over 1.7 s; beating it on the jaw, where the action is decisive, is harder than on position.

**ACT from Segment 3** gets exactly the same test: a new chunk from the real cameras every 50 steps, keeping the first 50 of its 60 steps. Its predictions were computed on the server's GPU with the Segment 3 checkpoint ([kdb6df/corl2026-suturebot-act](https://huggingface.co/kdb6df/corl2026-suturebot-act)). Two things to keep in mind when you compare:
- **Needle throw (clips 1441–1444): ACT is missing an input.** This checkpoint was trained with insertion and exit points drawn on the endoscope image, and those are not in the Open-H data. So on needle throws ACT runs without its targets, as in Segment 3 ("diagnostic only"). Knot tying (1445–1451) needs no dots, so it's the fair comparison.
- **ACT's training clips weren't recorded**, so it may have seen some of these test clips; GR00T-H did not.

In [ ]:
#@title Open-loop MSE / MAE (GR00T-H vs ACT) { display-mode: "form" }
ACT_PRED_URL = ""  #@param {type:"string"}
# optional: your own ACT from Segment 3, a (T,16) .npy URL with the same 50-step protocol; replaces the precomputed ACT
r = _post("/v1/groot/open_loop", {"episode": EPISODE, "start": 0, "steps": 10_000}, timeout=600) if BACKEND == "api" else BUNDLE["open_loop"]
gt, pred = np.array(r["gt"]), np.array(r["pred"])
GROUPS = {"xyz (mm)": [0, 1, 2, 8, 9, 10], "quat": [3, 4, 5, 6, 11, 12, 13, 14], "jaw (rad)": [7, 15]}
def ol_metrics(gt, pred):  # MSE/MAE == open_loop_eval.py; per-group MAE added for readability
    pred = np.array(pred, copy=True)
    for s in (slice(3, 7), slice(11, 15)):  # q and -q are the same rotation: align signs first
        pred[np.sum(gt[:, s] * pred[:, s], 1) < 0, s] *= -1
    m = {"MSE": float(np.mean((gt - pred) ** 2)), "MAE": float(np.mean(np.abs(gt - pred)))}
    for g, ix in GROUPS.items():
        m[f"MAE {g}"] = float(np.mean(np.abs(gt[:, ix] - pred[:, ix]))) * (1000 if "mm" in g else 1)
    return m
hold = gt[np.arange(len(gt)) // 50 * 50]   # "do nothing": repeat the pose at each chunk start
rows = {"GR00T-H-N1.7": ol_metrics(gt, pred), "hold pose (no model)": ol_metrics(gt, hold)}
act = None
if ACT_PRED_URL:   # (T,16) .npy from Segment 3, same episode, same 50-step protocol
    act = np.load(io.BytesIO(requests.get(ACT_PRED_URL).content))[: len(gt)]
elif ACT is not None:
    act = ACT["open_loop"][: len(gt)]
if act is not None:
    rows["ACT (Segment 3)" + (" ⚠️ no goal dots" if TASKS[int(df.task_index.iloc[0])] == "needle throw" else "")] = ol_metrics(gt[: len(act)], act)
display(pd.DataFrame(rows).T.style.format("{:.4f}"))

fig, ax = plt.subplots(2, 4, figsize=(16, 5), sharex=True)
for i, d in enumerate([0, 1, 2, 7, 8, 9, 10, 15]):
    a = ax.flat[i]; a.plot(scale(d) * gt[:, d], "k", lw=1, label="GT"); a.plot(scale(d) * pred[:, d], "C1", lw=1, label="GR00T-H")
    if act is not None: a.plot(scale(d) * act[:, d], "C2", lw=1, label="ACT")
    for s in range(0, len(gt), 50): a.axvline(s, color="r", lw=.3)
    a.set_title(NAMES[d], fontsize=9); a.set_ylabel(unit(d), fontsize=8)
    if i >= 4: a.set_xlabel("step (30 Hz)", fontsize=8)
ax.flat[0].legend(); fig.suptitle("Open-loop rollout (red lines = new chunk predicted)"); plt.tight_layout()

## ▶ 4 · What would those actions *do*? Ask Cosmos-H-Dreams (≈3 min)
Cosmos-H-Dreams takes **one frame + a sequence of robot actions** and generates the video that should follow, 12 frames (~1.2 s) at a time.
We roll out 6 blocks (72 frames, ~7 s, the protocol of the Open-H report) twice from the same start frame:
1. with the **recorded actions** → any difference from the real video is *world-model error*;
2. with **GR00T-H's actions** → differences on top of that come from the *policy*;
3. with **ACT's actions** (same receding-horizon rule from the same frame) → the same question for Segment 3's policy.

This is **open loop**: GR00T-H's chunks are computed from the *real* video, so Cosmos' generated frames never feed back into the policy (Section 5 closes the loop).

Scoring = per-frame L1 and SSIM against the real video (Open-H report, Methods).

In [ ]:
#@title Generate (or fetch) the rollouts { display-mode: "form" }
BLOCKS = 6
def dream(source, actions=None, frame=None, episode=None, instruction=None):
    if BACKEND != "api":
        print("🔌 This cell generates a new video, so it needs the live server (BACKEND = 'api')."); return None
    j = {"episode": episode or EPISODE, "frame": FRAME if frame is None else frame,
         "action_source": source, "blocks": BLOCKS, "instruction": instruction}
    if actions is not None: j["actions"] = np.asarray(actions).tolist()
    job = _post("/v1/cosmos/dream", j)
    t0 = time.time()
    while job.get("status") not in ("done", "error"):
        time.sleep(3); job = {**_get(f"/v1/jobs/{job['job_id']}").json(), "job_id": job["job_id"]}
        print(f"\r{source}: {job['status']} ({time.time()-t0:.0f}s)", end="")
    if job["status"] == "error": raise RuntimeError(job.get("error"))
    path = f"dream_{source}_{job['job_id']}.mp4"; open(path, "wb").write(_get(job["video"]).content)
    print(f"\r{source}: done{' (cached)' if job.get('cached') else ''}            "); return path

if BACKEND == "api":
    DREAM = {s: dream(s) for s in ("gt", "groot")}
    if ACT is not None and int(ACT["start_frame"]) == FRAME:   # pre-computed for each clip's start frame
        DREAM["act"] = dream("custom", actions=ACT["dream"])
else:
    DREAM = {s: "bundle/" + p for s, p in BUNDLE["dreams"].items()}

def load_dream(p):
    fr = read_frames(p)
    return fr[1:] if len(fr) == BLOCKS * 12 + 1 else fr   # drop context frame if included
real = read_frames(vid, idx=set(range(FRAME + 3, FRAME + 3 * BLOCKS * 12 + 1, 3)))
gen = {k: load_dream(p) for k, p in DREAM.items()}
n = min(len(real), *map(len, gen.values()))
if "act" in gen:   # 2 x 2: real | recorded  over  GR00T-H | ACT
    panel = np.concatenate([np.concatenate([real[:n], gen["gt"][:n]], axis=2),
                            np.concatenate([gen["groot"][:n], gen["act"][:n]], axis=2)], axis=1)
    print("top: real video | Cosmos + recorded actions      bottom: Cosmos + GR00T-H actions | Cosmos + ACT actions")
    show_video(to_mp4(panel, "side_by_side.mp4"), width=960)
else:
    panel = np.concatenate([real[:n], gen["gt"][:n], gen["groot"][:n]], axis=2)
    print("left: real video | middle: Cosmos + recorded actions | right: Cosmos + GR00T-H actions")
    show_video(to_mp4(panel, "side_by_side.mp4"), width=960)

In [ ]:
#@title Per-frame L1 / SSIM { display-mode: "form" }
from skimage.metrics import structural_similarity as ssim
def video_metrics(g, r):
    g, r = g.astype(np.float32) / 255, r.astype(np.float32) / 255
    l1 = np.abs(g - r).mean(axis=(1, 2, 3))
    s = np.array([ssim(a, b, channel_axis=-1, data_range=1.0) for a, b in zip(g, r)])
    return l1, s
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2)); summary = {}
for k, lab in [("gt", "recorded actions (world-model error)"), ("groot", "GR00T-H actions (+ policy error)"),
               ("act", "ACT actions (+ policy error)")]:
    if k not in gen: continue
    l1, s = video_metrics(gen[k][:n], real[:n]); summary[lab] = {"mean L1 ↓": l1.mean(), "mean SSIM ↑": s.mean()}
    ax[0].plot(l1, label=lab); ax[1].plot(s, label=lab)
for a, t in zip(ax, ["L1", "SSIM"]):
    a.set_title(t); a.set_xlabel("generated frame (~10 fps)"); [a.axvline(b * 12, color="gray", lw=.3) for b in range(BLOCKS)]
ax[0].legend(fontsize=8); plt.tight_layout(); display(pd.DataFrame(summary).T.style.format("{:.3f}"))

## ▶ 5 · Close the loop: GR00T-H drives inside the world model (≈3 min)
In sections 3 and 4 GR00T-H always saw the **real** video, so its mistakes never came back to it. Now the world model becomes the policy's simulator:

```
real frame at the start ──► GR00T-H ──► 50-step action chunk
         ▲                                   │ first 34 steps (1.1 s)
         │                                   ▼
 last generated frame  ◄── Cosmos-H-Dreams: 12 new frames (1.2 s)
```
Each turn of the loop is one 12-frame Cosmos block, and we repeat it to the end of the clip (up to 20 turns, 24 s). From the second turn on, **nothing real goes in**: GR00T-H only sees frames Cosmos made from GR00T-H's own actions. This is how NVIDIA screened GR00T-H checkpoints before running them on a robot (Cosmos-Surg-dVRK).

Two details make it work:
- **Endoscope only.** Cosmos renders the endoscope view, not the two wrist cameras. GR00T-H has an endoscope-only mode for this robot (`jhu_imerse_dvrk_mono`). It is the same checkpoint, just a different input setup. The table below shows its open-loop error next to the three-camera mode from section 3.
- **Where is the arm?** GR00T-H predicts motion *relative to the current arm pose*, and a generated video has no sensor reading. So each turn starts from the pose GR00T-H commanded on the previous turn ("assume the robot tracks perfectly").

**Why not ACT too?** ACT needs both wrist cameras as well as the endoscope, and Cosmos only renders the endoscope. Blank wrist images would be inputs it never saw, and real wrist frames would leak the real future, so ACT stays in the open-loop comparison of sections 3–4.

As in section 4 there are two rollouts:
1. **recorded actions**, restarted every 1.2 s from Cosmos' own last frame the same way → the *world model's* drift;
2. **GR00T-H in the loop** → drift on top of that comes from the *policy* reacting to generated frames.

In [ ]:
#@title Fetch (or run) the two closed loops { display-mode: "form" }
def closed_loop(policy, iterations=20, frame=None, instruction=None):
    """-> (video path, commanded actions (36*iterations, 16) at 30 Hz, start frame), or None"""
    frame = FRAME if frame is None or frame < 0 else frame
    if BACKEND != "api":
        c = (BUNDLE.get("closed_loop") or {}).get(policy) if (instruction, frame) == (None, FRAME) else None
        if not c:
            print("🔌 Not in the pre-computed results, so this needs the live server (BACKEND = 'api')."); return None
        return "bundle/" + c["video"], np.load("bundle/" + c["actions"]), c["frame"]
    job = _post("/v1/closed_loop", {"episode": EPISODE, "frame": frame, "iterations": iterations,
                                    "policy": policy, "instruction": instruction})
    its, t0 = job["iterations"], time.time()
    if not job.get("cached"): print(f"{policy}: queued (position {job.get('position')}), {its} turns of the loop")
    while job.get("status") not in ("done", "error"):
        time.sleep(2); job = {**_get(f"/v1/jobs/{job['job_id']}").json(), "job_id": job["job_id"]}
        print(f"\r{policy}: {job['status']}, turn {job.get('iteration', 0)}/{its} ({time.time()-t0:.0f}s)   ", end="")
    if job["status"] == "error": raise RuntimeError(job.get("error"))
    path = f"closed_{policy}_{job['job_id']}.mp4"; open(path, "wb").write(_get(job["video"]).content)
    acts = np.load(io.BytesIO(_get(job["actions"]).content))
    print(f"\r{policy}: done{' (cached)' if job.get('cached') else ''}, {its} turns = {its * 1.2:.1f} s of video        ")
    return path, acts, job.get("frame", frame)

def closed_loop_frames(path, f0):
    """generated frames (context dropped) and the matching real frames (generated frame j = raw frame f0 + 3j)"""
    g = read_frames(path)[1:]
    r = read_frames(vid, idx=set(range(f0 + 3, f0 + 3 * len(g) + 1, 3)))
    n = min(len(g), len(r)); return g[:n], r[:n]

CL = {p: closed_loop(p) for p in ("recorded", "groot")}
if all(CL.values()):
    F0 = CL["groot"][2]
    cl = {p: closed_loop_frames(v[0], F0)[0] for p, v in CL.items()}
    real_cl = closed_loop_frames(CL["groot"][0], F0)[1]
    n_cl = min(len(real_cl), *map(len, cl.values()))
    panel = np.concatenate([real_cl[:n_cl], cl["recorded"][:n_cl], cl["groot"][:n_cl]], axis=2)
    print("left: real video | middle: Cosmos + recorded actions, restarted every 1.2 s | right: GR00T-H ↔ Cosmos closed loop")
    show_video(to_mp4(panel, "closed_loop.mp4"), width=960)

In [ ]:
#@title How fast does it drift? { display-mode: "form" }
from skimage.metrics import structural_similarity as ssim
def video_metrics(g, r):   # same as section 4
    g, r = g.astype(np.float32) / 255, r.astype(np.float32) / 255
    l1 = np.abs(g - r).mean(axis=(1, 2, 3))
    return l1, np.array([ssim(a, b, channel_axis=-1, data_range=1.0) for a, b in zip(g, r)])

# 1. open-loop check first: is the endoscope-only mode as good as the three-camera one?
mono = (_post("/v1/groot/open_loop_mono", {"episode": EPISODE, "start": 0, "steps": 10_000}, timeout=600)
        if BACKEND == "api" else BUNDLE.get("open_loop_mono"))
if mono:
    g3 = _post("/v1/groot/open_loop", {"episode": EPISODE, "start": 0, "steps": 10_000}, timeout=600) if BACKEND == "api" else BUNDLE["open_loop"]
    gt_ol = np.array(g3["gt"])
    rows = {"GR00T-H, 3 cameras (section 3)": ol_metrics(gt_ol, np.array(g3["pred"])),
            "GR00T-H, endoscope only (used in the loop)": ol_metrics(np.array(mono["gt"]), np.array(mono["pred"])),
            "hold pose (no model)": ol_metrics(gt_ol, gt_ol[np.arange(len(gt_ol)) // 50 * 50])}
    print("Open loop, whole clip (real frames every 50 steps):")
    display(pd.DataFrame(rows).T[["MAE xyz (mm)", "MAE quat", "MAE jaw (rad)"]].style.format("{:.3f}"))

if all(CL.values()):
    # 2. image drift: both loops against the real video
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.2)); summary = {}
    for k, lab in [("recorded", "recorded actions (world-model drift)"), ("groot", "GR00T-H in the loop (+ policy drift)")]:
        l1, s = video_metrics(cl[k][:n_cl], real_cl[:n_cl]); summary[lab] = {"mean L1 ↓": l1.mean(), "mean SSIM ↑": s.mean()}
        ax[0].plot(l1, label=lab); ax[1].plot(s, label=lab)
    for a, t in zip(ax, ["L1 vs real video", "SSIM vs real video"]):
        a.set_title(t); a.set_xlabel("generated frame (~10 fps)"); [a.axvline(b * 12, color="gray", lw=.3) for b in range(n_cl // 12 + 1)]
    ax[0].legend(fontsize=8); plt.tight_layout(); plt.show(); display(pd.DataFrame(summary).T.style.format("{:.3f}"))

    # 3. motion drift: what GR00T-H commanded vs what the surgeon did
    A = CL["groot"][1]; R = GT[F0:F0 + len(A)]; A = A[:len(R)]; t = (F0 + np.arange(len(A))) / 30
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
    for j, (lab, dims) in enumerate([("PSM1 position", [0, 1, 2]), ("PSM2 position", [8, 9, 10]), ("jaws", JAWS)]):
        for i, d in enumerate(dims):
            ax[j].plot(t, scale(d) * R[:, d], "-", color=f"C{i}", lw=1, alpha=.5)
            ax[j].plot(t, scale(d) * A[:, d], "--", color=f"C{i}", lw=1.5, label=NAMES[d])
        for b in range(0, len(A), 36): ax[j].axvline((F0 + b) / 30, color="gray", lw=.3)
        ax[j].set_title(lab); ax[j].set_xlabel("time (s)"); ax[j].set_ylabel(unit(dims[0])); ax[j].legend(fontsize=7)
    fig.suptitle("solid = recorded (real surgeon) · dashed = GR00T-H's commands in the closed loop · grey = new turn of the loop")
    plt.tight_layout(); plt.show()
    err = 1000 * np.abs(A[:, POS] - R[:, POS]).mean(1)
    per_turn = [err[b:b + 36].mean() for b in range(0, len(err), 36)]
    print("mean position gap to the recorded motion, per 1.2 s turn (mm):", " ".join(f"{e:.1f}" for e in per_turn))

**Reading the results.** Both Cosmos rollouts drift away from the real video, and the recorded-actions one shows how much of that is the world model alone: it restarts from its own slightly blurry frame every 1.2 s. The GR00T-H loop adds the policy's own errors. A gap in the trajectory plot isn't automatically a failure, because a surgeon could do the same step another way. What matters is whether the arms keep doing something sensible on frames the policy has never seen before. Cosmos results vary a little from run to run, so read the curves for trends, not exact numbers. To score this properly you would judge task success (did the needle go through?) over many starts, which is how NVIDIA used Cosmos to rank checkpoints.

## ▶ 6 · Takeaways (≈1.5 min)
- **Pretraining helps** — in the Open-H report, GR00T-H matched ACT with 1/3 of the SutureBot data and was the only policy to complete end-to-end suturing (25 %).
- **GR00T-H vs ACT, open loop: close, and both near "hold still".** Across the 11 held-out clips, three-camera GR00T-H and ACT land at about 1.2–2.0 mm, and neither reliably beats holding the pose for 1.7 s. GR00T-H's endoscope-only mode (section 5) is clearly lower, at 0.7–1.1 mm. Short open-loop windows on slow surgical motion separate policies poorly, which is why the Open-H report also scores task success on the robot.
- **Open-loop error is not task success.** Low MSE can still fail on the robot; small errors compound once the policy sees its own consequences.
- **World models let us *look* at a policy's actions without a robot**, but they have their own error. Always roll out recorded actions as a reference.
- **A world model can also close the loop.** The policy then sees the consequences of its own actions, so its errors compound, which open-loop scores hide. The world model drifts too, so compare against the recorded-actions loop.
- **Next steps:** score closed loops by task success (e.g. did the needle go through?), simulation in [ORBIT-Surgical](https://orbit-surgical.github.io/), and fine-tuning GR00T-H on your own embodiment ([GR00T-H repo](https://github.com/NVIDIA-Medtech/GR00T-H)).
- **Run it yourself afterwards:** the same server is a one-click Brev Launchable (link on the tutorial website).

📄 Open-H-Embodiment consortium, *Open-H-Embodiment* report ([arXiv:2604.21017](https://arxiv.org/abs/2604.21017)).

---
# 🧪 Explore on your own
Each cell below stands alone (it only needs Setup and the clip above). Collapse this section during the live walkthrough.
Requests that are not pre-computed run live on the shared GPU and wait in a queue, so be patient with Cosmos jobs (~1 min each).

### 🧪 A · Change the instruction: does language steer the policy?
The first three choices are SutureBot's own task phrases, the only ones GR00T-H saw for this robot. The rest are off-script, and you can type your own. Expect small changes: the image tells the policy most of what to do, so this tests how much it listens to language at all.

In [ ]:
#@title Same frame, different instruction { display-mode: "form" }
INSTRUCTION = "knot tying"  #@param ["needle pickup", "needle throw", "knot tying", "do nothing", "move both arms out of view", "cut the suture", "open both jaws"] {allow-input: true}
if BACKEND == "cache":
    alt = BUNDLE["act_alt"].get(INSTRUCTION) or print("not in cache; needs the API")
else:
    alt = _post("/v1/groot/act", {"episode": EPISODE, "frame": FRAME, "instruction": INSTRUCTION})
if alt:
    A = np.array(alt["actions"])
    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    for j, (lab, dims) in enumerate([("PSM1 position", [0, 1, 2]), ("PSM2 position", [8, 9, 10]), ("jaws", JAWS)]):
        for k in dims:
            col = f"C{dims.index(k)}"   # one color per axis: solid = dataset instruction, dashed = yours
            ax[j].plot(scale(k) * CHUNK[:, k], "-", color=col, label=f"{NAMES[k]} (dataset)")
            ax[j].plot(scale(k) * A[:, k], "--", color=col, label=f"{NAMES[k]} ({INSTRUCTION})")
        ax[j].set_title(lab); ax[j].set_xlabel("step (30 Hz)"); ax[j].set_ylabel(unit(dims[0])); ax[j].legend(fontsize=6)
    dpos = 1000 * np.abs(A[:, POS] - CHUNK[:, POS]).mean(); djaw = np.abs(A[:, JAWS] - CHUNK[:, JAWS]).mean()
    plt.suptitle(f"dataset instruction vs {INSTRUCTION!r}: mean |Δ| position {dpos:.2f} mm, jaw {djaw:.3f} rad")
    plt.tight_layout()

### 🧪 B · Dream the instruction-steered actions

In [ ]:
#@title Cosmos rollout of a different instruction (live) { display-mode: "form" }
INSTRUCTION_B = "knot tying"  #@param ["needle pickup", "needle throw", "knot tying", "do nothing", "move both arms out of view", "cut the suture", "open both jaws"] {allow-input: true}
p = dream("groot", instruction=INSTRUCTION_B)
show_video(p)

### 🧪 C · Break the actions on purpose: is the world model paying attention?
Each choice edits the *recorded* actions, then Cosmos-H-Dreams renders them next to the real video and the unedited rollout. If the world model follows the actions, the scene should go visibly wrong. Positions are in metres in the data, so 0.01 = 1 cm. Some choices (e.g. *swap arms*, *5x the motion*) push Cosmos so far outside what it was trained on that the video smears or dissolves into noise: that is the world model failing, not a bug.

In [ ]:
#@title Perturb recorded actions and dream them (live) { display-mode: "form" }
PERTURBATION = "reverse the motion"  #@param ["freeze PSM1", "open both jaws", "snap both jaws open/shut", "swap arms", "double the motion", "5x the motion", "reverse the motion", "shift PSM1 by 1 cm in x", "shake both arms (±3 mm noise)", "twist PSM1 wrist 90°"]
n_raw = BLOCKS * 36 + 1
a = GT[FRAME:FRAME + n_raw].copy(); a0 = a[0].copy()
ramp = np.clip(np.arange(len(a)) / 15, 0, 1)[:, None]   # ease offsets in over 0.5 s instead of jumping

def qmul(p, q):  # Hamilton product, quaternions stored xyzw like the dataset
    x1, y1, z1, w1 = p.T; x2, y2, z2, w2 = q.T
    return np.stack([w1*x2 + x1*w2 + y1*z2 - z1*y2, w1*y2 - x1*z2 + y1*w2 + z1*x2,
                     w1*z2 + x1*y2 - y1*x2 + z1*w2, w1*w2 - x1*x2 - y1*y2 - z1*z2], 1)

if PERTURBATION == "freeze PSM1":            a[:, 0:8] = a0[0:8]
elif PERTURBATION == "open both jaws":       a[:, JAWS] = GT[:, JAWS].max(0)
elif PERTURBATION == "snap both jaws open/shut":   # full open <-> full shut every 0.5 s
    a[:, JAWS] = np.where((np.arange(len(a)) // 15 % 2)[:, None] == 0, GT[:, JAWS].max(0), GT[:, JAWS].min(0))
elif PERTURBATION == "swap arms":            a = np.concatenate([a[:, 8:], a[:, :8]], 1)
elif PERTURBATION == "double the motion":    a[:, POS] = a0[POS] + 2 * (a[:, POS] - a0[POS])
elif PERTURBATION == "5x the motion":        a[:, POS] = a0[POS] + 5 * (a[:, POS] - a0[POS])
elif PERTURBATION == "reverse the motion":   a[:, POS] = a0[POS] - (a[:, POS] - a0[POS])   # same path, opposite direction
elif PERTURBATION == "shift PSM1 by 1 cm in x": a[:, [0]] += 0.01 * ramp
elif PERTURBATION == "shake both arms (±3 mm noise)":
    a[:, POS] += np.random.default_rng(0).normal(0, 0.003, (len(a), len(POS)))
elif PERTURBATION == "twist PSM1 wrist 90°":  # rotate about the tool's own z-axis, eased in over 0.5 s
    th = np.deg2rad(90) * ramp[:, 0]
    a[:, 3:7] = qmul(a[:, 3:7], np.stack([0 * th, 0 * th, np.sin(th / 2), np.cos(th / 2)], 1))

ref = GT[FRAME:FRAME + n_raw]
move = max(np.linalg.norm(a[:, s] - ref[:, s], axis=1).max() for s in (slice(0, 3), slice(8, 11)))   # per arm, metres
rot = max(np.degrees(2 * np.arccos(np.clip(np.abs(np.sum(a[:, s] * ref[:, s], 1)), 0, 1))).max() for s in (slice(3, 7), slice(11, 15)))
print(f"{PERTURBATION}: largest change vs recorded: {1000 * move:.1f} mm position, {rot:.0f}° rotation, "
      f"{np.abs(a[:, JAWS] - ref[:, JAWS]).max():.2f} rad jaw")
p = dream("custom", actions=a)
if p:
    bad = load_dream(p); m = min(n, len(bad))
    print("left: real video | middle: Cosmos + recorded actions | right: Cosmos +", PERTURBATION)
    show_video(to_mp4(np.concatenate([real[:m], gen["gt"][:m], bad[:m]], axis=2), "perturbed.mp4"), width=960)

### 🧪 D · Another episode

In [ ]:
#@title Pick any episode hosted on the server { display-mode: "form" }
if BACKEND == "api":
    eps = _get("/v1/episodes").json(); print(eps["episodes"], eps.get("meta", ""))
OTHER_EPISODE = 0  #@param {type:"integer"}
OTHER_FRAME = 0    #@param {type:"integer"}
if OTHER_EPISODE:
    r = _post("/v1/groot/open_loop", {"episode": OTHER_EPISODE, "start": OTHER_FRAME, "steps": 300}, timeout=600)
    print("open-loop:", r["metrics"])

### 🧪 E · Bring your own policy (e.g. your ACT from Segment 3)

In [ ]:
#@title Dream any (T,16) absolute action array @30 Hz { display-mode: "form" }
MY_ACTIONS_NPY = ""  #@param {type:"string"}
# a local path or a URL; left empty, this dreams ACT's predictions from section 3 (ACT_PRED_URL)
src = MY_ACTIONS_NPY or ACT_PRED_URL
if src:
    mine = np.load(io.BytesIO(requests.get(src).content)) if src.startswith("http") else np.load(src)
    print("open-loop vs GT:", ol_metrics(GT[:len(mine)], mine))
    show_video(dream("custom", actions=mine[FRAME:FRAME + BLOCKS * 36 + 1]))

### 🧪 F · Run your own closed loop (live)
Pick a start frame, an instruction and how many turns of the loop (1.2 s each), and let GR00T-H drive inside Cosmos-H-Dreams. The instruction choices are the same as in Explore A: does an off-script instruction still produce sensible motion once the policy sees its own consequences? This needs the live server. Closed loops share one queue, so the cell prints your place in it; each turn takes about 2 s of GPU time.

In [ ]:
#@title GR00T-H ↔ Cosmos closed loop (live) { display-mode: "form" }
TURNS = 5                                   #@param {type:"slider", min:1, max:20, step:1}
START = -1                                  #@param {type:"integer"}
INSTRUCTION_F = "(the clip's own instruction)"  #@param ["(the clip's own instruction)", "needle pickup", "needle throw", "knot tying", "do nothing", "move both arms out of view", "cut the suture", "open both jaws"] {allow-input: true}
if BACKEND != "api":
    print("🔌 This cell runs a new closed loop, so it needs the live server (BACKEND = 'api').")
elif "closed_loop" not in globals():
    print("Run the first cell of section 5 once to define closed_loop().")
else:
    instr = None if INSTRUCTION_F.startswith("(") else INSTRUCTION_F
    res = closed_loop("groot", iterations=int(TURNS), frame=int(START), instruction=instr)
    if res:
        g, r = closed_loop_frames(res[0], res[2])
        print(f"start frame {res[2]} | instruction: {instr or 'clip default'} | left: real video, right: closed loop")
        show_video(to_mp4(np.concatenate([r, g], axis=2), "my_closed_loop.mp4"), width=800)
        A = res[1]; R = GT[res[2]:res[2] + len(A)]; A = A[:len(R)]
        print("mean position gap to the recorded motion per turn (mm):",
              " ".join(f"{1000 * np.abs(A[b:b+36][:, POS] - R[b:b+36][:, POS]).mean():.1f}" for b in range(0, len(A), 36)))